# Whole-image StarDist segmentation at pyramid level 4

This proof of concept deliberately avoids the tissue grid. It loads the complete DAPI image at pyramid level 4, segments every StarDist instance, measures each detected nucleus separately, explores individual-nucleus cluster counts, and produces a per-nucleus coloured mask.

**Important:** level 4 is downsampled relative to level 0. This experiment tests whole-image coverage and cell-focused output, but level-4 nuclei may be too small for optimal StarDist accuracy. The final production pipeline should use level-0 chunks if this test shows loss of nuclear detail.

## 1. Imports and GPU check

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from csbdeep.utils import normalize
from matplotlib.colors import ListedColormap
from mxtifffile import MxTiffFile
from skimage.measure import regionprops_table
from sklearn.cluster import MiniBatchKMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from stardist.models import StarDist2D

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Configuration
Change only `QPTIFF_PATH` initially. All output files are written to `OUTPUT_DIR`.

In [ ]:
QPTIFF_PATH = Path('/full/path/to/your/image.qptiff')
OUTPUT_DIR = Path('./level4_whole_image_poc')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHANNEL = 'DAPI'
LEVEL = 4
MODEL_NAME = '2D_versatile_fluo'
PROB_THRESH = 0.35
NMS_THRESH = 0.40
N_TILES = (8, 8)

# Individual-nucleus clustering, not window clustering.
K_MIN = 3
K_MAX = 12
FIXED_K = None       # Example: set to 8 to force eight nuclear phenotypes.
FIT_SAMPLE = 200_000
SILHOUETTE_SAMPLE = 20_000
RANDOM_SEED = 42

## 3. Open QPTIFF and resolve the DAPI channel

In [ ]:
slide = MxTiffFile(str(QPTIFF_PATH))

def resolve_channel(slide, requested):
    if isinstance(requested, int):
        return requested
    requested = str(requested).strip().lower()
    for key in ('biomarker', 'fluorophore'):
        for info in slide.channel_info:
            value = info.get(key)
            if value and str(value).strip().lower() == requested:
                return int(info['index'])
    raise ValueError(f'Channel {requested!r} not found. Available: {slide.channel_info}')

def squeeze_channel(array):
    array = np.asarray(array)
    if array.ndim == 3 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 2:
        raise ValueError(f'Expected a single 2-D channel, received {array.shape}')
    return array

layer = resolve_channel(slide, CHANNEL)
level0_shape = tuple(map(int, slide.series[0].levels[0].pages[0].shape[:2]))
level4_shape = tuple(map(int, slide.series[0].levels[LEVEL].pages[0].shape[:2]))
print('DAPI layer:', layer)
print('Level-0 shape:', level0_shape)
print(f'Level-{LEVEL} shape:', level4_shape)
print('Approximate uint16 image memory: %.2f GB' % (np.prod(level4_shape) * 2 / 1e9))
print('Approximate float32 working-image memory: %.2f GB' % (np.prod(level4_shape) * 4 / 1e9))

## 4. Load the complete level-4 DAPI image
Unlike the main pipeline, this cell does not create or apply a tissue mask. Every level-4 pixel is presented to StarDist.

In [ ]:
dapi = squeeze_channel(slide.read_region(layer, level=LEVEL))
assert dapi.shape == level4_shape, (dapi.shape, level4_shape)
dapi_normalized = normalize(dapi.astype(np.float32), 1, 99.8, axis=(0, 1))
print('Loaded:', dapi.shape, dapi.dtype)

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(dapi_normalized, cmap='gray')
ax.set_title(f'Complete DAPI image at pyramid level {LEVEL}')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Run StarDist across the whole level-4 image
`n_tiles` limits inference memory but still returns one global label image. Increase it to `(12, 12)` or `(16, 16)` if this cell runs out of memory.

In [ ]:
model = StarDist2D.from_pretrained(MODEL_NAME)
labels, details = model.predict_instances(
    dapi_normalized,
    prob_thresh=PROB_THRESH,
    nms_thresh=NMS_THRESH,
    n_tiles=N_TILES,
)
n_detected = int(labels.max())
print('Detected StarDist instances:', n_detected)
np.savez_compressed(OUTPUT_DIR / 'level4_stardist_labels.npz', labels=labels)

## 6. Whole-image instance-mask overlay
Each StarDist instance receives a random display colour. This is segmentation, not phenotype clustering.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
instance_colours = np.zeros((n_detected + 1, 4), dtype=np.float32)
instance_colours[1:, :3] = rng.uniform(0.15, 1.0, size=(n_detected, 3))
instance_colours[1:, 3] = 0.55
instance_rgba = instance_colours[labels]

fig, ax = plt.subplots(figsize=(14, 14))
ax.imshow(dapi_normalized, cmap='gray')
ax.imshow(instance_rgba)
ax.set_title(f'All level-{LEVEL} StarDist instances: {n_detected:,}')
ax.axis('off')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'level4_all_instance_masks.png', dpi=200, bbox_inches='tight')
plt.show()
del instance_rgba

## 7. Measure every detected nucleus individually
No nucleus QC is applied. Every StarDist instance is retained.

In [ ]:
properties = [
    'label', 'area', 'perimeter', 'centroid', 'eccentricity', 'solidity',
    'major_axis_length', 'minor_axis_length', 'orientation',
    'mean_intensity', 'max_intensity'
]
cells = pd.DataFrame(regionprops_table(
    labels, intensity_image=dapi_normalized, properties=properties
))
cells = cells.rename(columns={
    'centroid-0': 'y_level4', 'centroid-1': 'x_level4',
    'mean_intensity': 'mean_dapi', 'max_intensity': 'max_dapi'
})
cells['circularity'] = np.where(
    cells['perimeter'] > 0,
    4 * np.pi * cells['area'] / np.square(cells['perimeter']), np.nan
)
cells['aspect_ratio'] = np.where(
    cells['minor_axis_length'] > 0,
    cells['major_axis_length'] / cells['minor_axis_length'], np.nan
)
cells['equivalent_diameter'] = 2 * np.sqrt(cells['area'] / np.pi)
scale_y = level0_shape[0] / level4_shape[0]
scale_x = level0_shape[1] / level4_shape[1]
cells['y_level0_approx'] = cells['y_level4'] * scale_y
cells['x_level0_approx'] = cells['x_level4'] * scale_x
cells.to_csv(OUTPUT_DIR / 'level4_individual_nuclei.csv', index=False)
print(cells[['area', 'equivalent_diameter', 'circularity', 'eccentricity', 'solidity', 'aspect_ratio']].describe())
median_diameter = float(cells['equivalent_diameter'].median())
print('Median detected diameter at level 4:', round(median_diameter, 2), 'pixels')
if median_diameter < 8:
    warnings.warn('Median nuclei are under 8 pixels wide at level 4. StarDist morphology may be unreliable; use level-0 chunking for the final analysis.')

## 8. Cluster individual nuclei
This is fundamentally different from the existing three **window** clusters. Here, one row and one cluster label correspond to one StarDist instance.

In [ ]:
MORPHOLOGY_FEATURES = [
    'area', 'circularity', 'eccentricity', 'solidity', 'aspect_ratio',
    'major_axis_length', 'minor_axis_length'
]
feature_table = cells[MORPHOLOGY_FEATURES].copy()
feature_table['area'] = np.log1p(feature_table['area'].clip(lower=0))
preprocess = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', RobustScaler()),
])
X = preprocess.fit_transform(feature_table)
rng = np.random.default_rng(RANDOM_SEED)
fit_index = rng.choice(len(X), min(len(X), FIT_SAMPLE), replace=False)
score_index = rng.choice(len(fit_index), min(len(fit_index), SILHOUETTE_SAMPLE), replace=False)
X_fit = X[fit_index]

candidate_k = [FIXED_K] if FIXED_K is not None else list(range(K_MIN, K_MAX + 1))
scores = []
models = {}
for k in candidate_k:
    model_k = MiniBatchKMeans(
        n_clusters=k, random_state=RANDOM_SEED, batch_size=4096, n_init=10
    )
    fit_labels = model_k.fit_predict(X_fit)
    silhouette = silhouette_score(X_fit[score_index], fit_labels[score_index])
    scores.append({'k': k, 'silhouette': float(silhouette)})
    models[k] = model_k

score_table = pd.DataFrame(scores).sort_values('k')
display(score_table)
selected_k = int(FIXED_K if FIXED_K is not None else score_table.loc[score_table['silhouette'].idxmax(), 'k'])
cell_model = models[selected_k]
cells['cell_cluster'] = cell_model.predict(X).astype(int)
print('Selected individual-cell k:', selected_k)
print(cells['cell_cluster'].value_counts().sort_index())
cells.to_csv(OUTPUT_DIR / 'level4_individual_nuclei_clustered.csv', index=False)
joblib.dump(
    {'features': MORPHOLOGY_FEATURES, 'preprocess': preprocess, 'model': cell_model},
    OUTPUT_DIR / 'level4_individual_cell_cluster_model.joblib'
)
score_table.to_csv(OUTPUT_DIR / 'level4_cell_cluster_selection.csv', index=False)

## 9. Plot cluster-count evidence
A larger number of cell clusters should be supported by separation in feature space, not selected merely because cells seem visually complex.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(score_table['k'], score_table['silhouette'], marker='o')
ax.axvline(selected_k, color='red', linestyle='--', label=f'selected k={selected_k}')
ax.set_xlabel('Number of individual-nucleus clusters')
ax.set_ylabel('Silhouette score')
ax.set_title('Individual-cell cluster-count comparison')
ax.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'level4_cell_cluster_selection.png', dpi=180)
plt.show()

## 10. Render the cell-focused phenotype mask
Every coloured object below is one StarDist instance, not a rectangular grid tile.

In [ ]:
COLOURS = [
    '#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9',
    '#F0E442', '#6A3D9A', '#B15928', '#1B9E77', '#FB8072', '#80B1D3'
]
label_to_cluster = np.zeros(n_detected + 1, dtype=np.int16)
label_to_cluster[cells['label'].to_numpy(int)] = cells['cell_cluster'].to_numpy(int) + 1
cluster_image = label_to_cluster[labels]
cmap = ListedColormap(['#000000'] + COLOURS[:selected_k])
rgba = cmap(cluster_image / selected_k)
rgba[..., 3] = np.where(cluster_image == 0, 0.0, 0.65)

fig, ax = plt.subplots(figsize=(14, 14))
ax.imshow(dapi_normalized, cmap='gray')
ax.imshow(rgba)
ax.set_title(f'Individual level-{LEVEL} nuclear phenotypes: k={selected_k}')
ax.axis('off')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'level4_individual_cell_cluster_overlay.png', dpi=200, bbox_inches='tight')
plt.show()

## 11. Summarise what each cell cluster means

In [ ]:
cluster_profiles = cells.groupby('cell_cluster')[MORPHOLOGY_FEATURES + ['equivalent_diameter']].median()
cluster_sizes = cells['cell_cluster'].value_counts().sort_index().rename('n_cells')
cluster_summary = cluster_profiles.join(cluster_sizes)
cluster_summary.to_csv(OUTPUT_DIR / 'level4_cell_cluster_summary.csv')
display(cluster_summary)

## Interpretation

- If previously uncoloured tissue now contains nuclear masks, the low-resolution tissue gate was responsible for missing regions in the main pipeline.
- If visible nuclei are still not segmented, level 4 is probably too downsampled or the StarDist probability threshold is too strict. Try `PROB_THRESH = 0.25`, but inspect false positives.
- If median detected nuclear diameter is below roughly 8 pixels, do not use level-4 morphology clusters as the final biological result. Use level-0 chunk segmentation and assemble a global per-nucleus overlay instead.
- More cell clusters are possible, but cluster count must be supported by morphology separation and representative-cell inspection.